# LeetCode 1265 — Print Immutable Linked List in Reverse bằng Python

Link đề: https://leetcode.com/problems/print-immutable-linked-list-in-reverse/ · Độ khó: **Medium** · Chủ đề: Linked List, Two Pointers, Stack, Recursion · 🔒 **Premium**

Skeleton LeetCode (theo lời giải Python trên doocs):

```python
# """
# This is the ImmutableListNode's API interface.
# You should not implement it, or speculate about its implementation.
# """
# class ImmutableListNode:
#     def printValue(self) -> None: # print the value of this node.
#     def getNext(self) -> 'ImmutableListNode': # return the next node.

class Solution:
    def printLinkedListInReverse(self, head: 'ImmutableListNode') -> None:
        ...
```

> **Ghi chú Premium:** đề bài trong notebook được viết lại theo hiểu biết về bài (tham khảo https://leetcode.doocs.org/en/lc/1265/). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.
>
> **Bài tương tác:** LeetCode cung cấp sẵn `ImmutableListNode`. Trong notebook ta tự dựng **bản giả lập** có `printValue()` / `getNext()`, cấm sửa nút, đếm số lần gọi API, và bắt phần in ra bằng `io.StringIO` + `contextlib.redirect_stdout` để kiểm tra.

## 1. Mục tiêu học

1. Làm việc với dữ liệu **chỉ truy cập qua API** (`getNext`, `printValue`) — không đọc giá trị, không sửa con trỏ.
2. Hiểu bản chất "in ngược" = **LIFO** ⇒ đệ quy (stack ngầm) hoặc stack tường minh, `O(n)` thời gian / `O(n)` bộ nhớ.
3. Giải hai câu hỏi follow-up: **bộ nhớ `O(1)`** (đánh đổi thời gian `O(n²)`) và **thời gian tuyến tính, bộ nhớ dưới tuyến tính** (chia căn `O(√n)`); thêm cách **chia để trị** `O(n log n)` / `O(log n)`.
4. Biết tự viết **lớp giả lập + bộ bắt output** để test bài tương tác.
5. Tránh lỗi: in trước khi đệ quy, lệch một khi đếm bước, tràn đệ quy với `n = 1000`, cố sửa danh sách.

## 2. Đề bài & ví dụ

Cho đầu `head` của một danh sách liên kết **bất biến** (immutable). Hãy **in ra giá trị của mọi nút theo thứ tự ngược**, chỉ dùng giao diện:

- `ImmutableListNode.printValue()`: in giá trị của nút hiện tại.
- `ImmutableListNode.getNext()`: trả về nút kế tiếp (`None` nếu hết).

Không được truy cập trực tiếp vào nút hay sửa danh sách. Input chỉ dùng để khởi tạo danh sách bên trong.

**Ví dụ 1:** `head = [1,2,3,4]` → in `[4,3,2,1]`.

**Ví dụ 2:** `head = [0,-4,-1,3,-5]` → `[-5,3,-1,-4,0]`.

**Ví dụ 3:** `head = [-2,0,6,4,4,-6]` → `[-6,4,4,6,0,-2]`.

**Ràng buộc:** độ dài trong `[1, 1000]`; giá trị trong `[-1000, 1000]`.

**Follow-up:** (1) bộ nhớ hằng số? (2) thời gian tuyến tính và bộ nhớ **dưới tuyến tính**?

In [1]:
import contextlib
import io
import math
import random
import sys
import time

class ImmutableListNode:
    """Bản GIẢ LẬP của LeetCode: chỉ có printValue() và getNext(), không cho sửa."""
    next_calls = 0            # đếm số lần gọi getNext (để đo thời gian theo số bước)
    print_calls = 0

    def __init__(self, val, nxt=None):
        object.__setattr__(self, "_val", val)
        object.__setattr__(self, "_next", nxt)

    def __setattr__(self, name, value):
        raise AttributeError("ImmutableListNode là bất biến: không được sửa nút")

    def printValue(self) -> None:
        ImmutableListNode.print_calls += 1
        print(self._val)

    def getNext(self) -> "ImmutableListNode":
        ImmutableListNode.next_calls += 1
        return self._next

def build_list(values):
    head = None
    for v in reversed(values):
        head = ImmutableListNode(v, head)
    return head

def capture(func, values):
    """Chạy func(head) và trả về danh sách giá trị đã được in ra (đúng định dạng output LeetCode)."""
    head = build_list(values)
    ImmutableListNode.next_calls = ImmutableListNode.print_calls = 0
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        func(head)
    return [int(line) for line in buf.getvalue().split()]

examples = [([1, 2, 3, 4], [4, 3, 2, 1]),
            ([0, -4, -1, 3, -5], [-5, 3, -1, -4, 0]),
            ([-2, 0, 6, 4, 4, -6], [-6, 4, 4, 6, 0, -2])]
print("Thử bộ bắt output với hàm in xuôi:", capture(lambda h: [h.printValue(), h.getNext().printValue()], [7, 8, 9]))

Thử bộ bắt output với hàm in xuôi: [7, 8]


## 3. Kiến thức nền

### 3.1 Danh sách liên kết đơn & duyệt bằng `getNext()`

Mỗi nút trỏ tới nút sau. Muốn đi hết danh sách: `node = head; while node: ...; node = node.getNext()`. Ta chỉ đi **được một chiều** (xuôi), không có con trỏ ngược.

In [2]:
head = build_list([1, 2, 3, 4])
node, steps = head, 0
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    while node:
        node.printValue()
        node = node.getNext()
        steps += 1
print("In xuôi:", buf.getvalue().split(), "| số nút:", steps)

In xuôi: ['1', '2', '3', '4'] | số nút: 4


### 3.2 Bất biến: không sửa được nút

Bản giả lập chặn mọi phép gán thuộc tính. Vì vậy các mẹo kiểu "đảo ngược danh sách rồi in" (bài 206) **không dùng được**.

In [3]:
try:
    head._next = None
except AttributeError as e:
    print("Không sửa được:", e)

Không sửa được: ImmutableListNode là bất biến: không được sửa nút


### 3.3 Bắt output bằng `redirect_stdout`

`printValue()` in ra màn hình. Để **kiểm tra tự động**, ta chuyển hướng `stdout` vào một `io.StringIO`, rồi tách các dòng thành số.

In [4]:
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print(5); print(-3)
print("Bắt được:", repr(buf.getvalue()), "->", [int(x) for x in buf.getvalue().split()])

Bắt được: '5\n-3\n' -> [5, -3]


### 3.4 Đệ quy & stack ngầm

Mỗi lời gọi hàm được đẩy lên **call stack**; lời gọi sau cùng kết thúc trước (LIFO). Nếu ta "đi tiếp rồi mới in", phần in sẽ chạy theo thứ tự ngược. Python giới hạn độ sâu ~1000 (`sys.getrecursionlimit()`), mà danh sách dài tới 1000 ⇒ cần chú ý.

In [5]:
def countdown(k):
    if k == 0:
        return
    countdown(k - 1)
    print(k, end=" ")
countdown(5); print()
print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())

1 2 3 4 5 
Giới hạn đệ quy mặc định: 1000


### 3.5 Stack tường minh bằng `list`

`append` để đẩy, `pop()` để lấy phần tử cuối — đều `O(1)`. Đẩy các nút theo thứ tự xuôi rồi `pop` dần sẽ được thứ tự ngược.

In [6]:
st = []
for x in [1, 2, 3]:
    st.append(x)
print([st.pop() for _ in range(len(st))])

[3, 2, 1]


### 3.6 Căn bậc hai nguyên `math.isqrt`

`math.isqrt(n)` = `⌊√n⌋`. Ta dùng nó để chia danh sách thành khoảng `√n` khối, mỗi khối khoảng `√n` nút.

In [7]:
for n in [1, 4, 10, 1000]:
    k = max(1, math.isqrt(n))
    print(f"n={n:4}: khối cỡ k={k}, số khối={math.ceil(n / k)}")

n=   1: khối cỡ k=1, số khối=1
n=   4: khối cỡ k=2, số khối=2
n=  10: khối cỡ k=3, số khối=4
n=1000: khối cỡ k=31, số khối=33


## 4. Ý tưởng nền tảng / trực giác

Danh sách `1 → 2 → 3 → 4`. Muốn in `4` trước, ta phải **đi tới cuối** trước; nhưng khi đã ở `4` thì không quay lại `3` được (không có con trỏ ngược). Có ba lối thoát:

1. **Nhớ lại đường đã đi** (stack / đệ quy): tốn `O(n)` bộ nhớ, `O(n)` thời gian.
2. **Không nhớ gì, đi lại từ đầu mỗi lần**: lần 1 đi 3 bước in `4`, lần 2 đi 2 bước in `3`… ⇒ `O(1)` bộ nhớ, `O(n²)` thời gian.
3. **Nhớ một phần** (mỗi `√n` nút nhớ một "cột mốc"): đi lại từ cột mốc gần nhất ⇒ `O(√n)` bộ nhớ, `O(n)` thời gian.

Tính tay cách 2 với `n = 4`: số bước `getNext` để định vị = `3 + 2 + 1 + 0 = 6 = n(n−1)/2`.

In [8]:
n = 4
print("Số bước theo cách 'đi lại từ đầu':", sum(range(n)), "= n(n-1)/2 =", n * (n - 1) // 2)
for n in [10, 100, 1000]:
    k = max(1, math.isqrt(n))
    print(f"n={n:4}: O(1) bộ nhớ ~ {n * (n - 1) // 2:7} bước | chia căn: ~{2 * n:5} bước, nhớ {math.ceil(n / k) + k:3} nút")

Số bước theo cách 'đi lại từ đầu': 6 = n(n-1)/2 = 6
n=  10: O(1) bộ nhớ ~      45 bước | chia căn: ~   20 bước, nhớ   7 nút
n= 100: O(1) bộ nhớ ~    4950 bước | chia căn: ~  200 bước, nhớ  20 nút
n=1000: O(1) bộ nhớ ~  499500 bước | chia căn: ~ 2000 bước, nhớ  64 nút


## 5. Cách 1 — Brute Force / baseline: bộ nhớ `O(1)`, thời gian `O(n²)`

**Ý tưởng:** đếm độ dài `n`. Với `i = n−1, n−2, …, 0`: đi từ `head` đúng `i` bước rồi `printValue()`.

Đây vừa là "brute force" (thời gian tệ) vừa trả lời **follow-up 1** (bộ nhớ hằng số).

In [9]:
def reverse_print_const(head):
    n, node = 0, head
    while node:
        n += 1
        node = node.getNext()
    for i in range(n - 1, -1, -1):
        node = head
        for _ in range(i):
            node = node.getNext()
        node.printValue()

for values, want in examples:
    got = capture(reverse_print_const, values)
    print(got, got == want, "| số lần getNext:", ImmutableListNode.next_calls)

[4, 3, 2, 1] True | số lần getNext: 10
[-5, 3, -1, -4, 0] True | số lần getNext: 15
[-6, 4, 4, 6, 0, -2] True | số lần getNext: 21


**Độ phức tạp:** `O(n²)` thời gian (`n + n(n−1)/2` lần `getNext`), **`O(1)`** bộ nhớ phụ.

**Nút thắt:** ta quên sạch đường đi và đi lại từ đầu mỗi lần. Nhớ toàn bộ đường đi thì `O(n)` thời gian nhưng `O(n)` bộ nhớ; nhớ **một phần** sẽ cho điểm cân bằng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — LIFO:** thứ tự in ngược với thứ tự gặp ⇒ cấu trúc LIFO (stack). Đệ quy "gọi tiếp rồi mới in" là stack ngầm.

**Insight 2 — đánh đổi thời gian/bộ nhớ:** nếu nhớ `m` "cột mốc" cách đều nhau, đoạn giữa hai cột mốc dài `≈ n/m`. In ngược một đoạn bằng stack cỡ `n/m`. Tổng bộ nhớ `m + n/m`, nhỏ nhất khi `m ≈ √n` ⇒ `O(√n)`; mỗi nút được đi qua `O(1)` lần ⇒ `O(n)` thời gian.

**Insight 3 — chia để trị:** in ngược đoạn `[L, R)` = in ngược nửa sau rồi in ngược nửa trước. Mỗi tầng đệ quy đi qua `O(n)` bước, có `log n` tầng ⇒ `O(n log n)` thời gian, `O(log n)` bộ nhớ.

**Quy tắc "nếu… thì…":**
- Nếu được dùng `O(n)` bộ nhớ ⇒ stack/đệ quy (đơn giản nhất).
- Nếu yêu cầu `O(1)` bộ nhớ ⇒ đi lại từ đầu, chấp nhận `O(n²)`.
- Nếu yêu cầu `O(n)` thời gian và bộ nhớ `o(n)` ⇒ chia căn.
- Nếu đệ quy sâu `n` tầng ⇒ cân nhắc `sys.setrecursionlimit` hoặc stack tường minh.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đệ quy (lời giải doocs)

`f(node)`: nếu `node` khác `None` thì gọi `f(node.getNext())` **rồi mới** `node.printValue()`.

In [10]:
def reverse_print_rec(head):
    if head:
        reverse_print_rec(head.getNext())
        head.printValue()

for values, want in examples:
    got = capture(reverse_print_rec, values)
    print(got, got == want)

[4, 3, 2, 1] True
[-5, 3, -1, -4, 0] True
[-6, 4, 4, 6, 0, -2] True


Bản nộp LeetCode (đúng như doocs). Với `n = 1000`, LeetCode chạy được; trong môi trường của bạn có thể cần nâng giới hạn đệ quy (xem Lỗi 3).

In [11]:
class SolutionRec:
    def printLinkedListInReverse(self, head: "ImmutableListNode") -> None:
        if head:
            self.printLinkedListInReverse(head.getNext())
            head.printValue()

print([capture(SolutionRec().printLinkedListInReverse, v) == w for v, w in examples])

[True, True, True]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(n)` bộ nhớ (call stack sâu `n`).

### 7.2 Cách 3 — Stack tường minh

Đẩy mọi nút vào `list`, rồi `pop` và `printValue()`. Giống hệt Cách 2 về độ phức tạp nhưng không lo giới hạn đệ quy.

In [12]:
class SolutionStack:
    def printLinkedListInReverse(self, head: "ImmutableListNode") -> None:
        stack = []
        while head:
            stack.append(head)
            head = head.getNext()
        while stack:
            stack.pop().printValue()

for values, want in examples:
    got = capture(SolutionStack().printLinkedListInReverse, values)
    print(got, got == want)

[4, 3, 2, 1] True
[-5, 3, -1, -4, 0] True
[-6, 4, 4, 6, 0, -2] True


**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Chia căn: `O(n)` thời gian, `O(√n)` bộ nhớ (follow-up 2)

1. Đếm `n`, chọn `k = ⌈√n⌉`.
2. Đi một lượt, lưu nút ở các vị trí `0, k, 2k, …` vào `checkpoints` (≈ `√n` nút).
3. Với các cột mốc **từ cuối về đầu**: từ cột mốc đó đi tối đa `k` nút (không vượt cột mốc sau), đẩy vào stack nhỏ, rồi in ngược stack.

Mỗi nút được `getNext` tối đa 3 lần (đếm, đặt cột mốc, in) ⇒ `O(n)`.

In [13]:
def reverse_print_sqrt(head):
    n, node = 0, head
    while node:
        n += 1
        node = node.getNext()
    k = max(1, math.isqrt(n - 1) + 1) if n > 0 else 1        # k = ceil(sqrt(n))
    checkpoints, node, i = [], head, 0
    while node:
        if i % k == 0:
            checkpoints.append(node)
        node = node.getNext()
        i += 1
    for start in reversed(checkpoints):
        block, node = [], start
        while node and len(block) < k:
            block.append(node)
            node = node.getNext()
        while block:
            block.pop().printValue()

for values, want in examples:
    got = capture(reverse_print_sqrt, values)
    print(got, got == want, "| getNext:", ImmutableListNode.next_calls)

[4, 3, 2, 1] True | getNext: 12
[-5, 3, -1, -4, 0] True | getNext: 15
[-6, 4, 4, 6, 0, -2] True | getNext: 18


Bản nộp LeetCode của Cách 4:

In [14]:
class SolutionSqrt:
    def printLinkedListInReverse(self, head: "ImmutableListNode") -> None:
        n, node = 0, head
        while node:
            n += 1
            node = node.getNext()
        k = 1
        while k * k < n:
            k += 1
        checkpoints, node = [], head
        for i in range(n):
            if i % k == 0:
                checkpoints.append(node)
            node = node.getNext()
        for start in reversed(checkpoints):
            block, node = [], start
            for _ in range(k):
                if node is None:
                    break
                block.append(node)
                node = node.getNext()
            for nd in reversed(block):
                nd.printValue()

print([capture(SolutionSqrt().printLinkedListInReverse, v) == w for v, w in examples])

[True, True, True]


**Độ phức tạp Cách 4:** `O(n)` thời gian (≤ `3n` lần `getNext`), `O(√n)` bộ nhớ (`⌈n/k⌉` cột mốc + khối cỡ `k`).

### 7.4 Cách 5 — Chia để trị: `O(n log n)` thời gian, `O(log n)` bộ nhớ

`rev(node, length)`: nếu `length == 1` in luôn; ngược lại đi `length // 2` bước tới `mid`, gọi `rev(mid, length − half)` (nửa sau) **rồi** `rev(node, half)` (nửa trước).

In [15]:
def reverse_print_dc(head):
    n, node = 0, head
    while node:
        n += 1
        node = node.getNext()

    def rev(node, length):
        if length == 0:
            return
        if length == 1:
            node.printValue()
            return
        half = length // 2
        mid = node
        for _ in range(half):
            mid = mid.getNext()
        rev(mid, length - half)     # nửa sau trước
        rev(node, half)             # rồi nửa trước

    rev(head, n)

for values, want in examples:
    got = capture(reverse_print_dc, values)
    print(got, got == want, "| getNext:", ImmutableListNode.next_calls)

[4, 3, 2, 1] True | getNext: 8
[-5, 3, -1, -4, 0] True | getNext: 10
[-6, 4, 4, 6, 0, -2] True | getNext: 13


**Độ phức tạp Cách 5:** mỗi tầng đệ quy đi tổng cộng ≤ `n/2` bước, có `⌈log₂ n⌉` tầng ⇒ `O(n log n)` thời gian; độ sâu đệ quy `O(log n)` ⇒ `O(log n)` bộ nhớ. Đây là điểm cân bằng thú vị giữa Cách 1 và Cách 4.

## 8. TRACE từng bước

Trace **Cách 4 (chia căn)** với `[10, 20, 30, 40, 50, 60, 70]` (`n = 7`, `k = 3`):

| Bước | Trạng thái |
|---|---|
| đếm | `n = 7`, `k = ⌈√7⌉ = 3` |
| cột mốc | vị trí 0, 3, 6 ⇒ nút `10, 40, 70` |
| khối từ 70 | stack `[70]` ⇒ in `70` |
| khối từ 40 | stack `[40, 50, 60]` ⇒ in `60, 50, 40` |
| khối từ 10 | stack `[10, 20, 30]` ⇒ in `30, 20, 10` |

In [16]:
def trace_sqrt(values):
    head = build_list(values)
    ImmutableListNode.next_calls = 0
    out = io.StringIO()
    n, node = 0, head
    while node:
        n += 1; node = node.getNext()
    k = 1
    while k * k < n:
        k += 1
    print(f"n = {n}, k = {k}")
    checkpoints, node = [], head
    for i in range(n):
        if i % k == 0:
            checkpoints.append(node)
        node = node.getNext()
    print("cột mốc:", [c._val for c in checkpoints], "(đọc _val chỉ để hiển thị trace)")
    for start in reversed(checkpoints):
        block, node = [], start
        for _ in range(k):
            if node is None: break
            block.append(node); node = node.getNext()
        with contextlib.redirect_stdout(out):
            for nd in reversed(block):
                nd.printValue()
        print(f"  khối từ {start._val}: stack = {[b._val for b in block]} -> in {[b._val for b in reversed(block)]}, bộ nhớ đang dùng = {len(checkpoints) + len(block)} nút")
    print("Output:", [int(x) for x in out.getvalue().split()], "| tổng getNext =", ImmutableListNode.next_calls)

trace_sqrt([10, 20, 30, 40, 50, 60, 70])

n = 7, k = 3
cột mốc: [10, 40, 70] (đọc _val chỉ để hiển thị trace)
  khối từ 70: stack = [70] -> in [70], bộ nhớ đang dùng = 4 nút
  khối từ 40: stack = [40, 50, 60] -> in [60, 50, 40], bộ nhớ đang dùng = 6 nút
  khối từ 10: stack = [10, 20, 30] -> in [30, 20, 10], bộ nhớ đang dùng = 6 nút
Output: [70, 60, 50, 40, 30, 20, 10] | tổng getNext = 21


In [17]:
def trace_dc(values):
    head = build_list(values)
    out = io.StringIO()
    n = len(values)
    def rev(node, length, depth):
        pad = "  " * depth
        seg = []
        x = node
        for _ in range(length):
            seg.append(x._val); x = x._next      # chỉ để hiển thị
        print(f"{pad}rev({seg})")
        if length == 1:
            with contextlib.redirect_stdout(out):
                node.printValue()
            print(f"{pad}  in {seg[0]}")
            return
        half = length // 2
        mid = node
        for _ in range(half):
            mid = mid.getNext()
        rev(mid, length - half, depth + 1)
        rev(node, half, depth + 1)
    rev(head, n, 0)
    print("Output:", [int(x) for x in out.getvalue().split()])

trace_dc([1, 2, 3, 4, 5])

rev([1, 2, 3, 4, 5])
  rev([3, 4, 5])
    rev([4, 5])
      rev([5])
        in 5
      rev([4])
        in 4
    rev([3])
      in 3
  rev([1, 2])
    rev([2])
      in 2
    rev([1])
      in 1
Output: [5, 4, 3, 2, 1]


## 9. Code nộp LeetCode

Bản chính dùng stack tường minh (an toàn mọi giới hạn đệ quy); bản đệ quy của doocs ở mục 7.1 cũng được chấp nhận.

In [18]:
# """
# This is the ImmutableListNode's API interface.
# You should not implement it, or speculate about its implementation.
# """
# class ImmutableListNode:
#     def printValue(self) -> None: # print the value of this node.
#     def getNext(self) -> 'ImmutableListNode': # return the next node.

class Solution:
    def printLinkedListInReverse(self, head: 'ImmutableListNode') -> None:
        stack = []
        while head:
            stack.append(head)
            head = head.getNext()
        while stack:
            stack.pop().printValue()

for values, want in examples:
    got = capture(Solution().printLinkedListInReverse, values)
    print(got, "OK" if got == want else "SAI")

[4, 3, 2, 1] OK
[-5, 3, -1, -4, 0] OK
[-6, 4, 4, 6, 0, -2] OK


## 10. Vì sao thuật toán đúng

**Stack (Cách 3).** Sau vòng 1, stack chứa các nút theo thứ tự `v₀, v₁, …, v_{n−1}` (đỉnh là `v_{n−1}`). `pop` lấy theo thứ tự ngược ⇒ in `v_{n−1}, …, v₀`.

**Đệ quy (Cách 2).** Quy nạp theo độ dài: danh sách rỗng in rỗng (đúng). Với danh sách `v₀ → rest`, hàm in ngược `rest` (giả thiết quy nạp) **rồi** in `v₀` ⇒ được `reverse(rest) + [v₀]` = `reverse(v₀ → rest)`.

**Bộ nhớ O(1) (Cách 1).** Lần lặp với chỉ số `i` in đúng nút thứ `i` (đi `i` bước từ đầu); `i` chạy từ `n−1` về `0`.

**Chia căn (Cách 4).** Các khối `[0,k), [k,2k), …` phủ kín và rời nhau; in các khối theo thứ tự từ khối cuối về khối đầu, trong mỗi khối in ngược ⇒ toàn bộ danh sách được in ngược.

**Chia để trị (Cách 5).** `reverse(A + B) = reverse(B) + reverse(A)`; quy nạp theo `length`.

## 11. Phân tích Time / Space

Gọi `n` = độ dài danh sách (≤ 1000). "Time" đếm theo số lần gọi API.

| Cách | Time | Space phụ | Follow-up |
|---|---|---|---|
| 1. Đi lại từ đầu | `O(n²)` | `O(1)` | ✔ bộ nhớ hằng |
| 2. Đệ quy | `O(n)` | `O(n)` (call stack) | |
| 3. Stack | `O(n)` | `O(n)` | |
| 4. Chia căn | `O(n)` | `O(√n)` | ✔ tuyến tính + dưới tuyến tính |
| 5. Chia để trị | `O(n log n)` | `O(log n)` | điểm cân bằng |

Ô dưới đây đo **thực tế** số lần `getNext` để xác nhận bảng.

In [19]:
def measure(func, n):
    head = build_list(list(range(n)))
    ImmutableListNode.next_calls = 0
    with contextlib.redirect_stdout(io.StringIO()):
        func(head)
    return ImmutableListNode.next_calls

print(f"{'n':>5} | {'O(1) mem':>9} | {'stack':>6} | {'sqrt':>6} | {'D&C':>6}")
for n in [10, 100, 1000]:
    row = [measure(f, n) for f in (reverse_print_const, Solution().printLinkedListInReverse,
                                   reverse_print_sqrt, reverse_print_dc)]
    print(f"{n:>5} | {row[0]:>9} | {row[1]:>6} | {row[2]:>6} | {row[3]:>6}")

    n |  O(1) mem |  stack |   sqrt |    D&C
   10 |        55 |     10 |     30 |     25
  100 |      5050 |    100 |    300 |    416


 1000 |    500500 |   1000 |   3000 |   5932


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

Ngoài kiểm tra output, ta kiểm tra **mỗi nút được in đúng một lần** (`print_calls == n`).

In [20]:
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(max(old_limit, 5000))      # cho bản đệ quy chạy được với n = 1000

ALL = {
    "const": reverse_print_const,
    "rec": reverse_print_rec,
    "rec_submit": SolutionRec().printLinkedListInReverse,
    "stack": SolutionStack().printLinkedListInReverse,
    "sqrt": reverse_print_sqrt,
    "sqrt_submit": SolutionSqrt().printLinkedListInReverse,
    "dc": reverse_print_dc,
    "submit": Solution().printLinkedListInReverse,
}
edge = [([5], [5]), ([1, 2], [2, 1]), ([0, 0, 0], [0, 0, 0]), ([-1000, 1000], [1000, -1000]),
        (list(range(9)), list(range(8, -1, -1))), (list(range(10)), list(range(9, -1, -1)))]
for name, f in ALL.items():
    for values, want in examples + edge:
        got = capture(f, values)
        assert got == want, (name, values, got)
        assert ImmutableListNode.print_calls == len(values), (name, "in thừa/thiếu")
print("Mọi cách đúng với", len(examples + edge), "test")

Mọi cách đúng với 9 test


### 12.2 Ngẫu nhiên (seed cố định) — mọi độ dài 1..120 và giá trị trong `[-1000, 1000]`

In [21]:
rng = random.Random(1265)
for n in range(1, 121):
    values = [rng.randint(-1000, 1000) for _ in range(n)]
    want = values[::-1]
    for name, f in ALL.items():
        assert capture(f, values) == want, (name, n)
print("Độ dài 1..120: mọi cách khớp values[::-1]")

Độ dài 1..120: mọi cách khớp values[::-1]


### 12.3 Giới hạn `n = 1000` + kiểm tra bộ nhớ của chia căn

Đo số nút được "giữ" cùng lúc trong Cách 4 bằng một phiên bản có đếm.

In [22]:
values = [rng.randint(-1000, 1000) for _ in range(1000)]
for name, f in ALL.items():
    t0 = time.perf_counter()
    assert capture(f, values) == values[::-1]
    print(f"{name:12}: {time.perf_counter() - t0:.3f}s, getNext = {ImmutableListNode.next_calls}")

def sqrt_peak_memory(n):
    k = 1
    while k * k < n:
        k += 1
    return math.ceil(n / k) + k
print("Bộ nhớ đỉnh (số nút) của chia căn với n=1000:", sqrt_peak_memory(1000), "so với stack:", 1000)

const       : 0.124s, getNext = 500500
rec         : 0.001s, getNext = 1000
rec_submit  : 0.001s, getNext = 1000
stack       : 0.001s, getNext = 1000
sqrt        : 0.001s, getNext = 3000
sqrt_submit : 0.001s, getNext = 3000
dc          : 0.002s, getNext = 5932
submit      : 0.001s, getNext = 1000
Bộ nhớ đỉnh (số nút) của chia căn với n=1000: 64 so với stack: 1000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — In trước khi đệ quy

Đảo thứ tự hai dòng ⇒ in **xuôi**.

In [23]:
def rec_wrong1(head):
    if head:
        head.printValue()                 # SAI: in trước
        rec_wrong1(head.getNext())

print("Sai :", capture(rec_wrong1, [1, 2, 3, 4]))
print("Đúng:", capture(Solution().printLinkedListInReverse, [1, 2, 3, 4]))

Sai : [1, 2, 3, 4]
Đúng: [4, 3, 2, 1]


### Lỗi 2 — Lệch một trong cách bộ nhớ `O(1)`

Đi `i + 1` bước (hoặc cho `i` chạy từ `n` xuống `1`) ⇒ bỏ sót phần tử đầu, và rơi vào `None` ở lần đầu. Ta thêm kiểm tra `None` để không văng lỗi mà vẫn thấy kết quả sai.

In [24]:
def const_wrong2(head):
    n, node = 0, head
    while node:
        n += 1; node = node.getNext()
    for i in range(n, 0, -1):             # SAI: i chạy n..1 thay vì n-1..0
        node = head
        for _ in range(i):
            node = node.getNext() if node else None
        if node:
            node.printValue()

print("Sai :", capture(const_wrong2, [1, 2, 3, 4]))
print("Đúng:", capture(reverse_print_const, [1, 2, 3, 4]))

Sai : [4, 3, 2]
Đúng: [4, 3, 2, 1]


### Lỗi 3 — Quên giới hạn đệ quy (n = 1000)

Python mặc định chỉ cho ~1000 tầng, và notebook/trình chạy đã chiếm sẵn vài chục tầng ⇒ đệ quy trên danh sách 1000 nút có thể **tràn**. Ta tạm hạ về giới hạn mặc định để minh hoạ, bắt lỗi rồi khôi phục.

In [25]:
sys.setrecursionlimit(1000)
try:
    got = capture(reverse_print_rec, list(range(1000)))
    print("Đệ quy chạy được, in", len(got), "giá trị")
except RecursionError as e:
    print("Đệ quy mặc định: RecursionError ->", str(e)[:45])
finally:
    sys.setrecursionlimit(max(old_limit, 5000))
print("Stack tường minh:", len(capture(Solution().printLinkedListInReverse, list(range(1000)))), "giá trị, không lỗi")

Đệ quy mặc định: RecursionError -> maximum recursion depth exceeded
Stack tường minh: 1000 giá trị, không lỗi


**Sửa:** dùng stack tường minh, hoặc `sys.setrecursionlimit(...)` trước khi đệ quy.

### Lỗi 4 — Cố đảo ngược danh sách (vi phạm "immutable")

Thói quen từ bài 206: đảo con trỏ rồi in. Với `ImmutableListNode` điều này bị cấm (bản giả lập ném `AttributeError`; trên LeetCode, cố truy cập trường nội bộ là vi phạm đề).

In [26]:
def reverse_then_print_wrong4(head):
    prev = None
    while head:
        nxt = head.getNext()
        head._next = prev                 # SAI: sửa danh sách bất biến
        prev, head = head, nxt
    while prev:
        prev.printValue()
        prev = prev.getNext()

try:
    print(capture(reverse_then_print_wrong4, [1, 2, 3]))
except AttributeError as e:
    print("Sai: bị chặn ->", e)
print("Đúng:", capture(Solution().printLinkedListInReverse, [1, 2, 3]))

Sai: bị chặn -> ImmutableListNode là bất biến: không được sửa nút
Đúng: [3, 2, 1]


### Lỗi 5 — Chia căn nhưng khối tràn sang khối sau

Quên giới hạn `len(block) < k` khi gom khối ⇒ mỗi khối kéo dài tới cuối danh sách, in **trùng lặp** nhiều lần.

In [27]:
def sqrt_wrong5(head):
    n, node = 0, head
    while node:
        n += 1; node = node.getNext()
    k = max(1, math.isqrt(n))
    checkpoints, node, i = [], head, 0
    while node:
        if i % k == 0:
            checkpoints.append(node)
        node = node.getNext(); i += 1
    for start in reversed(checkpoints):
        block, node = [], start
        while node:                        # SAI: không dừng ở k nút
            block.append(node); node = node.getNext()
        while block:
            block.pop().printValue()

print("Sai :", capture(sqrt_wrong5, [1, 2, 3, 4, 5]))
print("Đúng:", capture(reverse_print_sqrt, [1, 2, 3, 4, 5]))

Sai : [5, 5, 4, 3, 5, 4, 3, 2, 1]
Đúng: [5, 4, 3, 2, 1]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"In/duyệt ngược" trên cấu trúc chỉ đi xuôi** ⇒ stack hoặc đệ quy (LIFO).
- **Follow-up về bộ nhớ** ⇒ nghĩ tới phổ đánh đổi: `O(1)`/`O(n²)` ↔ `O(log n)`/`O(n log n)` ↔ `O(√n)`/`O(n)` ↔ `O(n)`/`O(n)`.
- **Chia căn (sqrt decomposition)**: lưu "cột mốc" mỗi `√n` phần tử — mẹo dùng nhiều trong bài khó (truy vấn đoạn, checkpointing).
- Bài tương tác: tự viết bản giả lập + **đếm số lần gọi API** để kiểm tra độ phức tạp, và **bắt stdout** để kiểm tra output.
- Đệ quy trong Python với độ sâu ~`n` ≥ 1000 ⇒ luôn nghĩ tới recursion limit.

## 15. Biến thể của bài

1. **LeetCode 206 — Reverse Linked List**: được phép sửa con trỏ ⇒ `O(1)` bộ nhớ, `O(n)` thời gian.
2. **In ngược từng khối `k` phần tử** (giữ thứ tự khối) — giống bài 25 nhưng chỉ in.
3. **In ngược chỉ các nút ở vị trí chẵn** hoặc chỉ `m` nút cuối.
4. **Phần tử thứ `k` tính từ cuối** (bài 19 / two pointers) ⇒ hai con trỏ cách nhau `k`, `O(1)` bộ nhớ.
5. **Checkpointing tổng quát**: với bộ nhớ `m` nút, có thể in ngược trong `O(n²/m)` thời gian.

Minh hoạ biến thể 4 (phần tử thứ `k` từ cuối, chỉ dùng API) và 5 (bộ nhớ tuỳ chọn `m`):

In [28]:
def print_kth_from_end(head, k):
    fast = slow = head
    for _ in range(k):
        fast = fast.getNext()
    while fast:
        fast, slow = fast.getNext(), slow.getNext()
    slow.printValue()

def reverse_print_with_memory(head, m):
    """In ngược khi chỉ được nhớ tối đa khoảng m nút: chia thành m khối, mỗi khối in bằng cách O(1)."""
    n, node = 0, head
    while node:
        n += 1; node = node.getNext()
    k = max(1, math.ceil(n / m))                # độ dài khối
    checkpoints, node = [], head
    for i in range(n):
        if i % k == 0:
            checkpoints.append((i, node))
        node = node.getNext()
    for start_idx, start in reversed(checkpoints):
        length = min(k, n - start_idx)
        for j in range(length - 1, -1, -1):     # trong khối: đi lại từ cột mốc
            nd = start
            for _ in range(j):
                nd = nd.getNext()
            nd.printValue()

print(capture(lambda h: print_kth_from_end(h, 2), [1, 2, 3, 4, 5]))
for m in [1, 3, 10]:
    got = capture(lambda h: reverse_print_with_memory(h, m), list(range(30)))
    print(f"m={m:2}: đúng={got == list(range(29, -1, -1))}, getNext={ImmutableListNode.next_calls}")

[4]
m= 1: đúng=True, getNext=495
m= 3: đúng=True, getNext=195
m=10: đúng=True, getNext=90


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đi lại từ đầu | Đếm `n`, mỗi lần đi `i` bước rồi in | `O(n²)` | `O(1)` | Follow-up bộ nhớ hằng |
| 2. Đệ quy | Gọi tiếp rồi mới in | `O(n)` | `O(n)` | Code ngắn nhất (doocs) |
| 3. Stack | Đẩy hết rồi pop | `O(n)` | `O(n)` | Mặc định, an toàn đệ quy |
| 4. Chia căn | Cột mốc mỗi `√n` nút, in ngược từng khối | `O(n)` | `O(√n)` | Follow-up tuyến tính + dưới tuyến tính |
| 5. Chia để trị | In nửa sau rồi nửa trước | `O(n log n)` | `O(log n)` | Bộ nhớ rất ít, chấp nhận thêm `log n` |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Hàm sau in gì với `[1, 2, 3]`?
```python
def mystery(h):
    if h:
        h.printValue()
        mystery(h.getNext())
        h.printValue()
```

### Bài 2 (tính tay)
Với `n = 10`, Cách 1 gọi `getNext` bao nhiêu lần (gồm cả lượt đếm)? Cách 4 (với `k = ⌈√n⌉`) giữ tối đa bao nhiêu nút cùng lúc?

### Bài 3 (điền TODO)
Hoàn thiện `print_reverse_blocks(head, k)`: in danh sách theo từng khối `k` nút, **mỗi khối in ngược** nhưng các khối giữ thứ tự (ví dụ `[1..5], k=2` ⇒ `2 1 4 3 5`).

### Bài 4 (tự viết lại)
Viết lại Cách 5 (chia để trị) **không đếm trước `n`**, bằng cách cho hàm nhận cặp `(start, end)` với `end` là nút ngay sau đoạn (`None` = hết danh sách): `reverse_print_dc_range(head)`.

### Bài 5 (mini challenge)
Viết `print_last_m_reversed(head, m)`: in ngược **chỉ `m` nút cuối** với bộ nhớ `O(m)` và chỉ **một** lượt duyệt (gợi ý: hàng đợi vòng / `collections.deque(maxlen=m)`).

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def print_reverse_blocks(head, k):
    # TODO: gom tối đa k nút vào block, in ngược block, lặp lại
    pass

# Bài 4 — TODO
def reverse_print_dc_range(head):
    def rev(start, end):
        # TODO: nếu đoạn rỗng thì dừng; nếu chỉ 1 nút thì in
        # TODO: tìm giữa bằng hai con trỏ chậm/nhanh dừng ở end
        pass
    rev(head, None)

# Bài 5 — TODO
def print_last_m_reversed(head, m):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) In khi "đi xuống" và khi "đi lên". (2) Đi xuống: `1 2 3`. (3) Đi lên: `3 2 1` ⇒ `1 2 3 3 2 1`.

**Bài 2.** (1) Lượt đếm: 10 lần. (2) Định vị: `0 + 1 + … + 9 = 45` ⇒ tổng **55**. (3) `k = 4`, số cột mốc `⌈10/4⌉ = 3`, khối ≤ 4 ⇒ tối đa **7** nút.

**Bài 3.** (1) Vòng ngoài khi `head` còn. (2) Vòng trong gom ≤ `k` nút. (3) `for nd in reversed(block): nd.printValue()`.

**Bài 4.** (1) Đoạn rỗng ⇔ `start is end`. (2) Một nút ⇔ `start.getNext() is end`. (3) Hai con trỏ: `slow`, `fast` cùng xuất phát ở `start`, `fast` nhảy 2 bước (dừng khi gặp `end`); gọi `rev(slow, end)` rồi `rev(start, slow)`.

**Bài 5.** (1) `deque(maxlen=m)` tự bỏ phần tử cũ. (2) Duyệt một lần, `append` mọi nút. (3) In ngược deque (`pop` từ phải).

## 19. Đáp án tham khảo

In [30]:
from collections import deque

def mystery(h):
    if h:
        h.printValue()
        mystery(h.getNext())
        h.printValue()

# Bài 1
assert capture(mystery, [1, 2, 3]) == [1, 2, 3, 3, 2, 1]
# Bài 2
assert measure(reverse_print_const, 10) == 55
assert sqrt_peak_memory(10) == 7

# Bài 3
def print_reverse_blocks_sol(head, k):
    while head:
        block = []
        while head and len(block) < k:
            block.append(head); head = head.getNext()
        for nd in reversed(block):
            nd.printValue()

# Bài 4
def reverse_print_dc_range_sol(head):
    def rev(start, end):
        if start is end:
            return
        if start.getNext() is end:
            start.printValue(); return
        slow = fast = start
        while fast is not end and fast.getNext() is not end:
            slow = slow.getNext()
            fast = fast.getNext().getNext()
        rev(slow, end)
        rev(start, slow)
    rev(head, None)

# Bài 5
def print_last_m_reversed_sol(head, m):
    if m <= 0:
        return
    window = deque(maxlen=m)
    while head:
        window.append(head); head = head.getNext()
    while window:
        window.pop().printValue()

def blocks_ref(values, k):
    out = []
    for i in range(0, len(values), k):
        out += values[i:i + k][::-1]
    return out

assert capture(lambda h: print_reverse_blocks_sol(h, 2), [1, 2, 3, 4, 5]) == [2, 1, 4, 3, 5]
rng = random.Random(9)
for _ in range(200):
    values = [rng.randint(-1000, 1000) for _ in range(rng.randint(1, 60))]
    k = rng.randint(1, 8); m = rng.randint(0, 70)
    assert capture(lambda h: print_reverse_blocks_sol(h, k), values) == blocks_ref(values, k)
    assert capture(reverse_print_dc_range_sol, values) == values[::-1]
    assert capture(lambda h: print_last_m_reversed_sol(h, m), values) == (values[::-1][:m] if m else [])
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template stack (mặc định):**

```python
stack = []
while head:
    stack.append(head)
    head = head.getNext()
while stack:
    stack.pop().printValue()
```

**Template chia căn (follow-up 2):** đếm `n` → `k = ⌈√n⌉` → lưu nút ở vị trí `0, k, 2k, …` → duyệt cột mốc từ cuối, gom ≤ `k` nút vào stack nhỏ, in ngược.

**Template bộ nhớ O(1) (follow-up 1):** đếm `n` → `for i in range(n-1, -1, -1)`: đi `i` bước từ `head`, `printValue()`.

**Checklist trước khi submit:**
- [ ] Chỉ dùng `getNext()` / `printValue()`; không đọc/sửa trường nội bộ.
- [ ] Đệ quy: **gọi tiếp trước, in sau**.
- [ ] Đệ quy sâu `n ≤ 1000` ⇒ cân nhắc stack tường minh hoặc `sys.setrecursionlimit`.
- [ ] Cách `O(1)`: `i` chạy `n−1 … 0`, đi đúng `i` bước.
- [ ] Chia căn: mỗi khối dừng ở `k` nút (không tràn sang khối sau).
- [ ] Mỗi nút in đúng một lần (test bằng giả lập đếm `print_calls`).
- [ ] Đúng chữ ký: `printLinkedListInReverse(self, head: 'ImmutableListNode') -> None`.

In [31]:
sys.setrecursionlimit(old_limit)   # trả lại giới hạn đệ quy ban đầu
print("Đã khôi phục recursion limit =", sys.getrecursionlimit())

Đã khôi phục recursion limit = 1000
